# Reconnaissance de l’écriture balinaise avec DeepLontar et ResNet18

Ce notebook explique le projet de bout en bout : provenance des données, prévention des fuites, préparation des caractères, entraînement GPU et interprétation des résultats. L’objectif actuel est la **classification de caractères isolés**. Une translittération de page complète nécessitera ensuite un détecteur, la remise en ordre des caractères et des règles linguistiques.

## 1. Résultat obtenu

Le modèle retenu est un ResNet18 préentraîné sur ImageNet, ajusté sur 54 classes DeepLontar. Sur 6 706 caractères issus de manuscrits absents de l’entraînement :

- accuracy : **96,76 %** ;
- macro-précision : **92,74 %** ;
- macro-rappel : **93,56 %** ;
- macro-F1 : **92,91 %**.

La macro-F1 donne le même poids à chaque classe. Elle est plus informative que l’accuracy lorsque certaines classes sont beaucoup plus fréquentes que d’autres.

## 2. Configuration locale

L’expérience a été exécutée sous Windows avec une RTX 5070 Laptop 8 Go. PyTorch 2.9.1 avec CUDA 12.8 est utilisé pour assurer la compatibilité avec l’architecture NVIDIA Blackwell. Le NPU Intel n’est pas utilisé : CUDA est mieux pris en charge par PyTorch pour cet entraînement CNN.

In [ ]:
import json
from pathlib import Path

import torch

print('PyTorch :', torch.__version__)
print('CUDA disponible :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU :', torch.cuda.get_device_name(0))
    print('CUDA PyTorch :', torch.version.cuda)

## 3. Dataset DeepLontar

DeepLontar est un dataset public de manuscrits balinais sur feuilles de palmier. Il contient 200 pages originales et 400 variantes produites par niveaux de gris et seuillage adaptatif. Les caractères sont décrits par des annotations YOLO : `classe x_centre y_centre largeur hauteur`, avec des coordonnées normalisées entre 0 et 1.

Téléchargement officiel : https://doi.org/10.6084/m9.figshare.20103803.v2

Les archives doivent être extraites sous `data/deeplontar/images` et `data/deeplontar/labels`.

## 4. Pourquoi un split aléatoire par image serait incorrect

Chaque page originale possède deux versions améliorées partageant exactement les mêmes caractères et coordonnées. Si une version va dans le train et une autre dans le test, le modèle voit presque le même exemple pendant l’apprentissage. L’accuracy devient alors artificiellement élevée.

Le script `prepare_deeplontar.py` regroupe les fichiers qui partagent les mêmes annotations. Un groupe complet est affecté à train, validation ou test. Les variantes améliorées sont conservées uniquement dans le train ; validation et test contiennent uniquement des pages originales.

In [ ]:
# Exécuter depuis la racine du dépôt :
# %run ../prepare_deeplontar.py

summary_path = Path('../data/processed/summary.json')
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print('Groupes de manuscrits :', summary['groups'])
    print('Échantillons :', summary['samples'])
    print('Classes retenues :', len(summary['retained_classes']))
    print('Classes exclues :', summary['excluded_classes'])
else:
    print('Préparez d’abord les données avec prepare_deeplontar.py')

### Contrôle des classes

Les annotations contiennent quelques IDs hors du schéma déclaré `0–54`. Ils sont ignorés. La classe 34 apparaît dans seulement deux manuscrits : il est impossible de la placer honnêtement dans train, validation et test. Elle est donc exclue de cette première expérience. Le dataset final contient 54 classes, 95 427 crops de train, 6 668 de validation et 6 706 de test.

## 5. Prétraitement

Chaque boîte YOLO est convertie en crop RGB avec 15 % de marge. Pour l’entraînement, le DataModule applique un recadrage aléatoire et une rotation légère, puis la normalisation ImageNet. Pour validation et test, le redimensionnement est déterministe. Les flips sont volontairement évités : retourner un glyph peut changer sa structure.

La taille 128×128 est suffisante pour les caractères isolés et accélère fortement l’entraînement par rapport à 224×224.

In [ ]:
import sys
sys.path.insert(0, '..')

from src.data import BalineseDataModule

data = BalineseDataModule(
    data_dir='../data/processed',
    backbone='resnet18',
    image_size=128,
    batch_size=128,
    num_workers=0,  # valeur robuste dans un notebook Windows
)
data.setup()
print('Classes :', data.num_classes)
print('Train / val / test :', len(data.train_dataset), len(data.val_dataset), len(data.test_dataset))

## 6. Architecture du modèle

ResNet18 utilise des connexions résiduelles qui facilitent l’optimisation. Les poids ImageNet fournissent des filtres visuels déjà utiles pour détecter contours et textures. La dernière couche est remplacée par une couche linéaire à 54 sorties.

L’optimiseur est Adam avec un learning rate de `3e-4`. La loss est l’entropie croisée. L’entraînement utilise AMP FP16 pour exploiter les Tensor Cores et réduire la mémoire GPU.

In [ ]:
from src.models.lightning_module import BalineseClassifier

model = BalineseClassifier(
    num_classes=data.num_classes,
    backbone='resnet18',
    learning_rate=3e-4,
    pretrained=True,
)
print(f'Paramètres : {sum(p.numel() for p in model.parameters()):,}')

## 7. Lancer l’entraînement

Pour éviter de bloquer l’interface du notebook sous Windows, l’expérience complète est lancée depuis un terminal à la racine du dépôt :

```powershell
.\.venv\Scripts\python.exe train.py --data-dir data/processed --backbone resnet18 --image-size 128 --batch-size 128 --num-workers 8 --learning-rate 0.0003 --max-epochs 8 --accelerator gpu --devices 1 --precision 16-mixed --checkpoint-dir models/checkpoints/resnet18-128
```

`ModelCheckpoint` conserve le minimum de `val_loss`. `EarlyStopping` arrête l’entraînement après cinq époques sans amélioration.

## 8. Interprétation de la courbe

Le meilleur checkpoint apparaît à l’époque d’indice 1 : `val_loss=0,1446` et `val_acc=97,59 %`. Aux époques suivantes, la loss de train continue de baisser mais la loss de validation augmente. C’est le signe d’un début de surapprentissage. Utiliser le dernier checkpoint aurait donc été moins bon que restaurer automatiquement le meilleur.

In [ ]:
metrics_path = Path('../reports/test_metrics.json')
metrics = json.loads(metrics_path.read_text())
for key in ('samples', 'accuracy', 'macro_precision', 'macro_recall', 'macro_f1'):
    print(f'{key:>16}: {metrics[key]:.4f}' if isinstance(metrics[key], float) else f'{key:>16}: {metrics[key]}')

## 9. Limites observées

L’accuracy globale est élevée, mais les classes rares restent fragiles. `class_50` ne possède que deux exemples de test et obtient un F1 nul. Cela ne prouve pas que l’architecture est mauvaise : le support est insuffisant pour mesurer ou apprendre cette classe. La prochaine collecte doit cibler en priorité les classes 20, 32, 50 et les autres classes ayant peu de manuscrits sources.

In [ ]:
worst = sorted(metrics['per_class'], key=lambda row: row['f1'])[:10]
for row in worst:
    print(row['class_name'], 'support=', row['support'], 'F1=', f"{row['f1']:.3f}")

## 10. De la classification à la translittération

Ce modèle suppose que le caractère est déjà localisé. Une application finale devra suivre ce pipeline :

1. détecter les caractères dans la page avec YOLO ;
2. recadrer chaque boîte ;
3. prédire sa classe avec ResNet18 ;
4. ordonner les boîtes selon les lignes et leur position ;
5. mapper les IDs vers les glyphes Unicode balinais ;
6. appliquer les règles de composition et produire la translittération latine.

Pour des lignes non segmentées, un CRNN avec CTC ou un Transformer OCR pourra être comparé à cette approche en deux étapes.

## Conclusion

Le résultat constitue une baseline reproductible et crédible : données officielles, séparation par source, test indépendant, métriques macro et checkpoint conservé selon la validation. La priorité scientifique suivante n’est pas d’agrandir immédiatement le réseau, mais d’améliorer les exemples des classes rares et de construire la correspondance classe → Unicode → translittération.